# Hi-C Tertiary Analysis — Interactive Notebook

This notebook walks through **every stage** of Hi-C tertiary analysis, starting from contact count matrices.

## Table of Contents
1. [Setup & Data Generation](#1-setup)
2. [Quality Control](#2-qc)
3. [Normalization](#3-norm)
4. [P(s) Distance Decay](#4-ps)
5. [A/B Compartments](#5-comp)
6. [TAD Analysis](#6-tads)
7. [Loop Analysis](#7-loops)
8. [Differential Analysis (A vs B)](#8-diff)
9. [Overview Figure](#9-overview)
10. [Biological Questions Answered](#10-questions)

In [ ]:
import sys, warnings, os
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from hic_tertiary.data.synthetic import generate_synthetic_hic, generate_condition_pair
from hic_tertiary.qc.metrics import run_qc
from hic_tertiary.normalization.methods import normalize, kr_normalize, vc_normalize
from hic_tertiary.distance_decay.ps_curve import ps_all_chromosomes, fit_power_law
from hic_tertiary.compartments.ab_calling import compartments_all_chromosomes
from hic_tertiary.tads.insulation import tad_analysis_all_chromosomes
from hic_tertiary.loops.enrichment import loop_analysis_all_chromosomes
from hic_tertiary.differential.comparison import differential_analysis
from hic_tertiary.plotting import figures as figs

print('All modules loaded successfully.')

## 1. Setup & Data Generation <a id='1-setup'></a>

We generate two synthetic Hi-C datasets (Condition A = control, Condition B = treatment).  
The synthetic data includes realistic:
- Power-law distance decay
- TAD block structure
- A/B compartment checkerboard
- Chromatin loop point enrichments
- Poisson sequencing noise

In [ ]:
RESOLUTION = 50_000   # 50 kb bins
N_BINS     = 200      # 200 bins = 10 Mb per chromosome
N_CHROM    = 3

data_a, data_b = generate_condition_pair(
    n_bins=N_BINS, resolution=RESOLUTION, seed_a=42, seed_b=99
)

chr_names  = data_a['chr_names']
CHROM      = chr_names[0]   # focus chromosome for single-chrom plots

mat_a_raw = data_a['matrices'][CHROM]
mat_b_raw = data_b['matrices'][CHROM]

print(f'Chromosomes : {chr_names}')
print(f'Matrix shape: {mat_a_raw.shape}')
print(f'Total contacts (A): {mat_a_raw.sum():,.0f}')
print(f'Total contacts (B): {mat_b_raw.sum():,.0f}')

### Visualize the raw contact matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, mat, label in zip(axes, [mat_a_raw, mat_b_raw], ['Condition A (raw)', 'Condition B (raw)']):
    from matplotlib.colors import LogNorm
    m = mat.copy().astype(float)
    m[m == 0] = np.nan
    im = ax.imshow(m, cmap='YlOrRd',
                   norm=LogNorm(vmin=1, vmax=np.nanpercentile(m, 98)),
                   aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.7, label='Contacts (log)')
    ax.set_title(label); ax.set_xlabel('Bin'); ax.set_ylabel('Bin')
plt.suptitle(f'Raw Hi-C Contact Maps — {CHROM}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Quality Control <a id='2-qc'></a>

In [ ]:
qc = run_qc(data_a)

print('=== Global QC ===')
g = qc['global']
print(f"  Cis fraction   : {g['cis_fraction']*100:.1f}%")
print(f"  Cis/trans ratio: {g['cis_trans_ratio']:.1f}")

print(f'\n=== {CHROM} QC ===')
c = qc[CHROM]
cov = c['coverage']
print(f"  Mean coverage  : {cov['mean']:.1f}")
print(f"  Zero bins      : {cov['n_zero_bins']} / {cov['total_bins']}")
comp = c['completeness']
print(f"  Completeness   : {comp['completeness']*100:.1f}%")

fig = figs.plot_qc_panel(mat_a_raw, qc, resolution=RESOLUTION, chrom=CHROM)
plt.show()

## 3. Normalization <a id='3-norm'></a>

Three normalization strategies are compared:
- **ICE** (Iterative Correction and Eigenvector decomposition) — makes the matrix doubly stochastic
- **KR** (Knight-Ruiz) — Newton-Raphson balancing, same mathematical goal as ICE
- **VC** (Vanilla Coverage) — simple sqrt(coverage) correction

In [ ]:
ice_a, bias_ice = normalize(mat_a_raw, method='ice')
kr_a,  bias_kr  = kr_normalize(mat_a_raw)
vc_a,  bias_vc  = vc_normalize(mat_a_raw)

print('ICE marginals (should be ~equal):', np.percentile(ice_a.sum(1)[ice_a.sum(1)>0], [5,50,95]).round(4))
print('KR  marginals (should be ~equal):', np.percentile( kr_a.sum(1)[ kr_a.sum(1)>0], [5,50,95]).round(4))

fig = figs.plot_normalization_comparison(mat_a_raw, ice_a, kr_a, vc_a, bias_ice)
plt.show()

In [ ]:
# Normalize all chromosomes for downstream analysis
norm_a, norm_b = {}, {}
for chrom in chr_names:
    norm_a[chrom], _ = normalize(data_a['matrices'][chrom], method='ice')
    norm_b[chrom], _ = normalize(data_b['matrices'][chrom], method='ice')

print('All chromosomes normalized.')

## 4. P(s) Distance Decay <a id='4-ps'></a>

**Question:** How does contact frequency decay with genomic distance?  
P(s) ~ s^{-α} where α ≈ 1.0 indicates a fractal globule (Lieberman-Aiden 2009).  
Transitions in the derivative reveal transitions between polymer regimes.

In [ ]:
ps_results = ps_all_chromosomes(norm_a, chr_names, RESOLUTION)

for chrom in chr_names:
    fit = ps_results[chrom]['fit']
    print(f"{chrom}: α={fit['alpha']:.3f}  A={fit['A']:.2f}  R²={fit['r_squared']:.3f}")

fig = figs.plot_ps_curves(ps_results)
plt.show()

## 5. A/B Compartments <a id='5-comp'></a>

**Question:** Which genomic regions are transcriptionally active (A) vs heterochromatic (B)?  

Method:
1. Compute O/E matrix (normalize away distance decay)
2. Take Pearson correlation of O/E rows
3. PCA: first PC = compartment eigenvector
4. Saddle plot: sort by PC1 → reveals A-A / B-B enrichment pattern

In [ ]:
comp_a = compartments_all_chromosomes(norm_a, chr_names)
comp_b = compartments_all_chromosomes(norm_b, chr_names)

for chrom in chr_names:
    c = comp_a[chrom]
    n_A = (c['labels'] == 1).sum()
    n_B = (c['labels'] == -1).sum()
    s   = c['strength']['strength']
    v   = c['var_explained'][0]
    print(f"{chrom}: {n_A} A-bins, {n_B} B-bins | PC1 explains {v*100:.1f}% var | strength={s:.2f}")

fig = figs.plot_compartment_map(norm_a[CHROM], comp_a[CHROM], RESOLUTION, CHROM)
plt.show()

## 6. TAD Analysis <a id='6-tads'></a>

**Questions:**
- Where are TAD boundaries?
- How large are TADs?
- Which genes are co-regulated (in the same TAD)?

Method: Diamond Insulation Score (Crane et al., Nature 2015)  
Local minima in the insulation score = TAD boundaries.

In [ ]:
tad_a = tad_analysis_all_chromosomes(norm_a, chr_names, RESOLUTION,
                                      window=10, delta_threshold=0.1, min_distance=3)
tad_b = tad_analysis_all_chromosomes(norm_b, chr_names, RESOLUTION,
                                      window=10, delta_threshold=0.1, min_distance=3)

for chrom in chr_names:
    t = tad_a[chrom]
    med_mb = np.median(t['tad_sizes_bp'])/1e6 if len(t['tad_sizes_bp']) else 0
    print(f"{chrom}: {len(t['tads'])} TADs | {len(t['boundaries'])} boundaries | median size={med_mb:.2f} Mb")

fig = figs.plot_tad_analysis(norm_a[CHROM], tad_a[CHROM], RESOLUTION, CHROM)
plt.show()

## 7. Loop Analysis <a id='7-loops'></a>

**Questions:**
- Which pairs of loci form stable chromatin loops?
- Are loop anchors at TAD boundaries (CTCF-CTCF loops)?
- What is the distribution of loop sizes?

APA (Aggregate Peak Analysis): stack sub-matrices centred on each loop → the average pile-up shows the enrichment signal.

In [ ]:
loop_a = loop_analysis_all_chromosomes(norm_a, chr_names, RESOLUTION,
                                        min_dist_bins=5, max_dist_bins=100,
                                        enrichment_threshold=2.0, flank=10)

for chrom in chr_names:
    l = loop_a[chrom]
    med_mb = np.median(l['loop_sizes'])/1e6 if len(l['loop_sizes']) else 0
    print(f"{chrom}: {len(l['loops'])} loops | APA enrichment={l['apa_score']:.2f}x | median size={med_mb:.2f} Mb")

fig = figs.plot_loop_analysis(loop_a[CHROM], RESOLUTION, CHROM)
plt.show()

## 8. Differential Analysis (A vs B) <a id='8-diff'></a>

**Questions:**
- Which genomic regions show significantly different contacts?
- Which bins switch A↔B compartment between conditions?
- Are TAD boundaries gained or lost?
- How similar are the two maps overall? (SCC)

In [ ]:
data_a_norm = dict(**data_a, matrices=norm_a)
data_b_norm = dict(**data_b, matrices=norm_b)

diff = differential_analysis(
    data_a_norm, data_b_norm,
    comp_a, comp_b,
    tad_a,  tad_b,
    pseudocount=1.0, scc_max_dist=100
)

print('=== Differential Summary ===')
for chrom in chr_names:
    d = diff[chrom]
    print(f"\n{chrom}:")
    print(f"  SCC            : {d['scc_result']['scc']:.3f}  (1.0 = identical maps)")
    print(f"  Compartment switches: {d['comp_diff']['n_switched']} bins")
    print(f"    A→B : {d['comp_diff']['n_a_to_b']} bins")
    print(f"    B→A : {d['comp_diff']['n_b_to_a']} bins")
    print(f"  Gained boundaries: {d['ins_diff']['gained'].sum()}")
    print(f"  Lost  boundaries: {d['ins_diff']['lost'].sum()}")

In [ ]:
fig = figs.plot_differential(
    diff[CHROM], diff[CHROM]['comp_diff'],
    chrom=CHROM, resolution=RESOLUTION
)
plt.show()

fig = figs.plot_scc_heatmap(diff)
plt.show()

## 9. Overview Figure <a id='9-overview'></a>

Everything on one figure: contact map + compartment track + insulation score + loop positions.

In [ ]:
fig = figs.plot_summary_overview(
    norm_a[CHROM], comp_a[CHROM], tad_a[CHROM], loop_a[CHROM],
    resolution=RESOLUTION, chrom=CHROM
)
plt.show()

## 10. Biological Questions Answered <a id='10-questions'></a>

### Questions Hi-C can answer

#### Genome organisation
| Question | Analysis | Result |  
|----------|----------|--------|
| Where is active chromatin? | A/B compartments (PC1 > 0) | `comp_a[chrom]['labels'] == 1` |
| Where is heterochromatin? | A/B compartments (PC1 < 0) | `comp_a[chrom]['labels'] == -1` |
| How large are TADs? | Insulation score | `tad_a[chrom]['tad_sizes_bp']` |
| Where are TAD boundaries? | Insulation score minima | `tad_a[chrom]['boundaries']` |

#### 3D structure
| Question | Analysis | Result |
|----------|----------|--------|
| What polymer model fits? | P(s) exponent α | `ps_results[chrom]['fit']['alpha']` |
| Which loci form loops? | Local O/E enrichment | `loop_a[chrom]['loops']` |
| How enriched are loops? | APA score | `loop_a[chrom]['apa_score']` |

#### Comparative / disease
| Question | Analysis | Result |
|----------|----------|--------|
| How similar are two maps? | SCC | `diff[chrom]['scc_result']['scc']` |
| Which bins switch A↔B? | Differential compartments | `diff[chrom]['comp_diff']['switched']` |
| Are boundaries gained/lost? | Differential insulation | `diff[chrom]['ins_diff']['gained']` |
| Which contacts change? | Log fold change map | `diff[chrom]['lfc']` |

In [ ]:
# ── Example: find the 5 most strongly enriched loops ─────────────────────────
all_loops = loop_a[CHROM]['loops']
sorted_loops = sorted(all_loops, key=lambda x: x['enrichment'], reverse=True)

print(f'Top 5 loops in {CHROM} (by O/E enrichment):')
print(f'{"Anchor 1":>10}  {"Anchor 2":>10}  {"Distance (Mb)":>15}  {"O/E":>8}')
print('-' * 50)
for lp in sorted_loops[:5]:
    dist_mb = (lp['j'] - lp['i']) * RESOLUTION / 1e6
    a1_mb   = lp['i'] * RESOLUTION / 1e6
    a2_mb   = lp['j'] * RESOLUTION / 1e6
    print(f"{a1_mb:>10.2f} Mb  {a2_mb:>7.2f} Mb  {dist_mb:>12.2f} Mb  {lp['enrichment']:>8.2f}x")

In [ ]:
# ── Example: bins that switched A→B between conditions ───────────────────────
switched = diff[CHROM]['comp_diff']

a_to_b_bins = np.where(switched['a_to_b'])[0]
b_to_a_bins = np.where(switched['b_to_a'])[0]

print(f'Bins switching A→B: {len(a_to_b_bins)} ({len(a_to_b_bins)/N_BINS*100:.1f}% of genome)')
print(f'Bins switching B→A: {len(b_to_a_bins)} ({len(b_to_a_bins)/N_BINS*100:.1f}% of genome)')
if len(a_to_b_bins):
    print('\nFirst A→B switches (Mb):', (a_to_b_bins[:5] * RESOLUTION / 1e6).round(2))

In [ ]:
# ── Save all figures to disk ──────────────────────────────────────────────────
import os
os.makedirs('../results/plots', exist_ok=True)

figs.plot_qc_panel(mat_a_raw, qc, RESOLUTION, CHROM, save_path='../results/plots/01_qc_panel.png'); plt.close('all')
figs.plot_normalization_comparison(mat_a_raw, ice_a, kr_a, vc_a, bias_ice, save_path='../results/plots/02_normalization_comparison.png'); plt.close('all')
figs.plot_ps_curves(ps_results, save_path='../results/plots/03_ps_curves.png'); plt.close('all')
figs.plot_compartment_map(norm_a[CHROM], comp_a[CHROM], RESOLUTION, CHROM, save_path=f'../results/plots/04_compartments_{CHROM}.png'); plt.close('all')
figs.plot_tad_analysis(norm_a[CHROM], tad_a[CHROM], RESOLUTION, CHROM, save_path=f'../results/plots/05_tads_{CHROM}.png'); plt.close('all')
figs.plot_loop_analysis(loop_a[CHROM], RESOLUTION, CHROM, save_path=f'../results/plots/06_loops_{CHROM}.png'); plt.close('all')
figs.plot_differential(diff[CHROM], diff[CHROM]['comp_diff'], CHROM, RESOLUTION, save_path=f'../results/plots/07_differential_{CHROM}.png'); plt.close('all')
figs.plot_scc_heatmap(diff, save_path='../results/plots/08_scc_summary.png'); plt.close('all')
figs.plot_summary_overview(norm_a[CHROM], comp_a[CHROM], tad_a[CHROM], loop_a[CHROM], RESOLUTION, CHROM, save_path=f'../results/plots/09_overview_{CHROM}.png'); plt.close('all')

import glob
saved = sorted(glob.glob('../results/plots/*.png'))
print(f'Saved {len(saved)} figures to results/plots/')
for s in saved:
    print(' ', os.path.basename(s))